[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/practical-biopython-course/blob/main/notebooks/03_seqrecord_seqio.ipynb)

# 03 · SeqRecord and SeqIO: Reading and Writing FASTA and GenBank

*Practical Biopython Course · Hamza Hentabli, Ph.D. · ~45 min*

Real sequences come with names, descriptions, and annotations. Biopython stores them in a
`SeqRecord`, and `Bio.SeqIO` reads and writes dozens of formats with one interface.

**Learning objectives.** By the end of this notebook you will be able to:

- explain what a `SeqRecord` holds (id, description, sequence, annotations, features)
- parse FASTA and GenBank files with `SeqIO.parse` / `SeqIO.read`
- index large files with `SeqIO.to_dict` and `SeqIO.index`
- filter, convert, and write sequence files

In [1]:
# Setup: installs dependencies on Google Colab and locates the course data.
# Running locally? Install requirements.txt once and this cell does nothing extra.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'biopython'], check=True)
    if not os.path.isdir("practical-biopython-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/practical-biopython-course.git"], check=True)
    DATA_DIR = "practical-biopython-course/data"
else:
    DATA_DIR = "../data" if os.path.isdir("../data") else "data"

print("Running on Colab:", IN_COLAB, "| data folder:", DATA_DIR)

Running on Colab: False | data folder: ../data


## 1. Reading a multi-record FASTA file

In [2]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd

records = list(SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta"))
print(len(records), "records")
egfr = records[0]
print("id:         ", egfr.id)
print("description:", egfr.description)
print("sequence:   ", egfr.seq[:50], "...", len(egfr), "aa")

15 records
id:          P00533|EGFR_HUMAN
description: P00533|EGFR_HUMAN Epidermal growth factor receptor OS=Homo sapiens GN=EGFR
sequence:    MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLS ... 1210 aa


In FASTA, everything up to the first space of the header becomes `id`; the whole header is `description`.

In [3]:
rows = []
for r in records:
    acc, entry = r.id.split("|")
    organism = r.description.split("OS=")[1].split(" GN=")[0]
    rows.append({"accession": acc, "entry": entry, "organism": organism, "length": len(r)})
pd.DataFrame(rows)

,accession,entry,organism,length
0,P00533,EGFR_HUMAN,Homo sapiens,1210
1,P04626,ERBB2_HUMAN,Homo sapiens,1255
2,P21860,ERBB3_HUMAN,Homo sapiens,1342
3,Q15303,ERBB4_HUMAN,Homo sapiens,1308
4,Q01279,EGFR_MOUSE,Mus musculus,1210
5,P04412,EGFR_DROME,Drosophila melanogaster,1426
6,P24348,LET23_CAEEL,Caenorhabditis elegans,1323
7,P00519,ABL1_HUMAN,Homo sapiens,1130
8,P12931,SRC_HUMAN,Homo sapiens,536
9,P15056,BRAF_HUMAN,Homo sapiens,766


## 2. Random access: `to_dict` and `index`

`SeqIO.to_dict` loads everything into memory, fine for small files. `SeqIO.index` keeps only file
offsets, so it works on multi-gigabyte files too.

In [4]:
by_id = SeqIO.to_dict(records)
print(by_id["P04626|ERBB2_HUMAN"].description)

idx = SeqIO.index(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta")
print(len(idx), "records indexed;", idx["P24941|CDK2_HUMAN"].seq[:20])
idx.close()

P04626|ERBB2_HUMAN Receptor tyrosine-protein kinase erbB-2 OS=Homo sapiens GN=ERBB2
15 records indexed; MENFQKVEKIGEGTYGVVYK


## 3. Reading a GenBank file

In [5]:
gb = SeqIO.read(f"{DATA_DIR}/entrez_cache/NM_005228.gb", "genbank")
print(gb.id, "|", gb.name, "|", len(gb), "nt")
print(gb.description)
for key in ["molecule_type", "topology", "date", "organism", "source"]:
    print(f"{key:>14}: {gb.annotations.get(key)}")
print("taxonomy:", " > ".join(gb.annotations["taxonomy"][-5:]))
print("cross-references:", gb.dbxrefs)

NM_005228.5 | NM_005228 | 9905 nt
Homo sapiens epidermal growth factor receptor (EGFR), transcript variant 1, mRNA
 molecule_type: mRNA
      topology: linear
          date: 13-APR-2026
      organism: Homo sapiens
        source: Homo sapiens (human)
taxonomy: Primates > Haplorrhini > Catarrhini > Hominidae > Homo
cross-references: []


In [6]:
print(len(gb.features), "features; first three:")
for f in gb.features[:3]:
    print(" ", f.type, f.location)

83 features; first three:
  source [0:9905](+)
  gene [0:9905](+)
  exon [0:349](+)


In [7]:
refs = gb.annotations["references"]
print(len(refs), "literature references; the first:")
print(refs[0].title, "|", refs[0].journal)

10 literature references; the first:
TLR9 signaling requires ligand-induced phosphorylation of two specific tyrosine residues by EGFR and Syk | mBio 16 (11), e0027625 (2025)


Notebook 04 explores the features in depth. Here, notice that GenBank carries far more than FASTA.

## 4. Building a SeqRecord yourself and writing files

In [8]:
import os
os.makedirs("output", exist_ok=True)

kinase = SeqRecord(egfr.seq[711:979], id="EGFR_kinase_712-979",
                   description="human EGFR kinase domain (UniProt P00533, residues 712-979)")
SeqIO.write(kinase, "output/egfr_kinase.fasta", "fasta")
print(open("output/egfr_kinase.fasta").read()[:200])

>EGFR_kinase_712-979 human EGFR kinase domain (UniProt P00533, residues 712-979)
FKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDN
PHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLE


Filtering with a generator expression keeps memory low. Here we keep only human proteins and write them.

In [9]:
human = (r for r in SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta")
         if "OS=Homo sapiens" in r.description)
n = SeqIO.write(human, "output/human_kinases.fasta", "fasta")
print(n, "human records written")

12 human records written


## 5. Converting formats

`SeqIO.convert` reads one format and writes another. GenBank → FASTA loses annotations; FASTA → GenBank
needs a `molecule_type` because FASTA does not store it.

In [10]:
n = SeqIO.convert(f"{DATA_DIR}/entrez_cache/NM_005228.gb", "genbank", "output/NM_005228_converted.fasta", "fasta")
print(n, "record converted")
back = SeqIO.read("output/NM_005228_converted.fasta", "fasta")
print("same sequence as GenBank:", back.seq == gb.seq)

1 record converted
same sequence as GenBank: True


In [11]:
kinase.annotations["molecule_type"] = "protein"
SeqIO.write(kinase, "output/egfr_kinase.gb", "genbank")
print(open("output/egfr_kinase.gb").read()[:300])

LOCUS       EGFR_kinase_712-979      268 aa                     UNK 01-JAN-1980
DEFINITION  human EGFR kinase domain (UniProt P00533, residues 712-979).
ACCESSION   EGFR_kinase_712-979
VERSION     EGFR_kinase_712-979
KEYWORDS    .
SOURCE      .
  ORGANISM  .
            .
FEATURES             Locati


### ✏️ Exercise 1

Write all panel proteins **shorter than 800 aa** to `output/short_kinases.fasta` and report how many
were written.

In [12]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
short = (r for r in SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta") if len(r) < 800)
print(SeqIO.write(short, "output/short_kinases.fasta", "fasta"), "records written")
```

</details>

### ✏️ Exercise 2

Create a new FASTA file containing only the kinase domains of the panel, using the start and end columns of
`data/sequences/kinase_panel.csv`. Give each record the id `<GENE>_<ORGANISM-GENUS>_kd`. Compare your result
with `data/sequences/kinase_domains.fasta`.

In [13]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
panel = pd.read_csv(f"{DATA_DIR}/sequences/kinase_panel.csv")
full = SeqIO.to_dict(SeqIO.parse(f"{DATA_DIR}/sequences/kinase_panel_full.fasta", "fasta"),
                     key_function=lambda r: r.id.split("|")[0])
kds = [SeqRecord(full[row.accession].seq[row.kinase_start - 1: row.kinase_end],
                 id=f"{row.gene}_{row.organism.split()[0]}_kd", description="")
       for row in panel.itertuples()]
SeqIO.write(kds, "output/my_kinase_domains.fasta", "fasta")
ref = [r.seq for r in SeqIO.parse(f"{DATA_DIR}/sequences/kinase_domains.fasta", "fasta")]
print("identical to the course file:", [k.seq for k in kds] == ref)
```

</details>

### ✏️ Exercise 3

From the GenBank record `gb`, print the first three references' authors (first 60 characters) and their
PubMed IDs.

In [14]:
# Your code here

<details>
<summary><b>Show solution</b></summary>

```python
for ref in gb.annotations["references"][:3]:
    print(ref.pubmed_id, "|", ref.authors[:60])
```

</details>

## Key takeaways

- `SeqRecord` = sequence + id + description + annotations + features.
- `SeqIO.read` for exactly one record, `SeqIO.parse` for many, `SeqIO.index` for big files.
- `SeqIO.write` and `SeqIO.convert` handle output; FASTA drops annotations.

**Next:** [04 · Sequence features](04_sequence_features_annotations.ipynb).